In [51]:
from anytree import NodeMixin, RenderTree

def print_tree(node):
    # print(RenderTree(node))
    for pre, fill, node in RenderTree(node):
        print("%s%s" % (pre, node))

In [58]:
class Rule(NodeMixin):
    _name: str

    @property
    def name(self):
        return getattr(self, '_name', None) or type(self).__name__

    def __call__(self, context):
        pass

    def __str__(self):
        return f"{self.name}()"

In [59]:
rule = Rule()
rule_1 = Rule()
rule.children += rule_1,
rule_1_1 = Rule()
rule_1_1.parent = rule_1
print_tree(rule)

Rule()
└── Rule()
    └── Rule()


In [60]:
type(rule.children)


tuple

In [61]:
class RuleRenderer:
    def __init__(self, rule):
        self.rule = rule

    def __iter__(self):
        return iter(RenderTree(self.rule))


In [62]:
for pre, fill, node in RuleRenderer(rule):
    print("%s%s" % (pre, node))

Rule()
└── Rule()
    └── Rule()


In [63]:
rule = Rule()
child = Rule()
child.parent = rule
renderer = RuleRenderer(rule)
for pre, _, node in renderer:
     print(f"{pre}{node.name}")

Rule
└── Rule


In [65]:
from parse import parse

In [66]:
res = parse("My name is {name}, {}", "My name is John, 29")
res

<Result ('29',) {'name': 'John'}>

In [67]:
res.fixed

('29',)

In [68]:
res.named

{'name': 'John'}

In [1]:
from anyrule.parsing import KeyMatchingDict

r = KeyMatchingDict()

def say_hello(age,name):
    print(f"Hello {name}, you are {age} years old")

r["My name is {name}, {age}."] = say_hello


In [2]:
result = r.match("My name is John, 29.")
result

KeyParsingValueResult(fixed=(), named={'name': 'John', 'age': '29'}, value=<function say_hello at 0x000001B2CE221BC0>)

In [4]:
result.execute_value()

Hello John, you are 29 years old


In [7]:

from types import SimpleNamespace


context = SimpleNamespace()
context.x = 1
delattr(context, "x")

```
if "alcholol" in item.tags and buyer.age < 18:
   deny_alchol
```

In [1]:
from types import SimpleNamespace
from typing import NamedTuple

class Buyer(NamedTuple):
    name: str
    age: int

class Item(NamedTuple):
    name: str
    price: float
    tags: list

underage_buyer = Buyer(name="John", age=15)
item = Item(name="item", price=10, tags=["alchohol"])

context = SimpleNamespace(buyer=underage_buyer, item=item)

In [2]:
from anyrule import Rule

class DenyRule(Rule):
    def __init__(self, reason):
        self.reason = reason
    
    def __call__(self, context):
        raise ValueError(f"Deny for reason: {self.reason}")
    
class SmileRule(Rule):
    def __call__(self, context):
        print("Smile")

In [3]:
from anyrule import RuleExecutor, IfEvalRule

if_rule = IfEvalRule("""("alchohol" in item.tags) and (buyer.age < 18)""")
then_rule = DenyRule("Underage")
else_rule = SmileRule()
if_rule.children += then_rule, else_rule

executor = RuleExecutor()
executor.execute(if_rule, context)


ValueError: Deny for reason: Underage

In [13]:
expr = """("alcholol" in item.tags) and (buyer.age < 18)"""
expr = """"alcholol" in item.tags"""
# expr = """(buyer.age < 18)"""
eval(expr, context.__dict__, {})

False

In [12]:
'alchohol' in context.item.tags

True

In [19]:
eval("'alchohol' in item.tags", {}, context.__dict__)

True

In [1]:
# The Domain Model

from types import SimpleNamespace
from typing import NamedTuple


class Buyer(NamedTuple):
    name: str
    age: int


class Item(NamedTuple):
    name: str
    price: float
    tags: list


# Custom rules
from anyrule import Rule


class DenyRule(Rule):
    def __init__(self, reason):
        self.reason = reason

    def __call__(self, context):
        print(f"Deny with reason: {self.reason}")


class SmileRule(Rule):
    def __call__(self, context):
        print("Smile")

# Business Rule definition
from anyrule import RuleExecutor, IfEvalRule

no_underage_alchohol_rule = IfEvalRule("""("alchohol" in item.tags) and (buyer.age < 18)""")
then_rule = DenyRule("Underage")
else_rule = SmileRule()
no_underage_alchohol_rule.children += then_rule, else_rule


underage_buyer = Buyer(name="John", age=15)
adult_buyer = Buyer(name="Jane", age=25)
wine = Item(name="Red wine", price=10, tags=["alchohol"])

context = SimpleNamespace(buyer=underage_buyer, item=wine)
executor = RuleExecutor()

# Adult
context.buyer = adult_buyer
executor.execute(no_underage_alchohol_rule, context)

# Adult
context.buyer = underage_buyer
executor.execute(no_underage_alchohol_rule, context)


Smile
Deny with reason: Underage
